# DATA ENGINE: recognizer v3 / teacher pilot

Wybierz GPU i uruchom wszystkie komorki. Dane pobiora sie automatycznie z HF; nie wgrywaj zadnych ZIP-ow. 64 wycinki pochodza ze zbioru treningowego, nie z 15 stron walidacji ani testu. Qwen i TrOCR pracuja kolejno w osobnych procesach, bez etykiet referencyjnych. Notebook nie trenuje modelu: przygotowuje propozycje do kontroli obrazu, granic linii i pisowni. Zgodnosc modeli nie stanowi prawdy ani zgody na trening. Zachowujemy historyczne znaki, bez modernizacji. Profil wycinkow nie zostal jeszcze sprawdzony na GPU. W razie bledow ostatnia komorka pobiera logi i czesciowe wyniki.

In [ ]:
import datetime
import json
import os
from pathlib import Path
import subprocess
import sys

CONFIG = {'schema': 'slayer-recognizer-data-v3-teacher-pilot-v1', 'status': 'frozen-before-GPU-inference', 'dataset': {'repo': 'PiotrSty/slayer-ocr-datasets', 'revision': '4735f045cc791df1926940a1fa0cd78c2c82b48c', 'path': 'data/recognizer-data-v3-pilot-v1-20261005/recognizer-data-v3-pilot-input.zip', 'archive_sha256': 'aa691ba4496709666842b6a44a29d8af08e816a83fbcf346fddf268fe777acbf', 'manifest_sha256': 'e89e83506f8179c4a10a76bd29bbf18bbce84cbd8211076307fbcd3cdfcff158', 'lines': 64, 'forbidden_collections': ['Ceremonie_FT', 'Choragiew_FT', 'Diariusz_FT', 'List_FT', 'NA2_FT', 'NA4_FT', 'Nowe_nowiny_z_Czech_FT', 'Nowiny_z_Inflant_FT', 'Nowiny_z_Rakuz_FT', 'Powodzenia_FT', 'Prawdziwa_relacja_FT', 'Relacja_chwalebnej_FT', 'Relacja_koronacji_FT', 'Relacja_prawdziwa_FT', 'Relacja_spraw_FT', 'SLAWNA_VICTORIA_FT', 'Slawna_wiktoria_FT', 'Wiesc_FT', 'Wyprawa_2_FT', 'Zalosne_FT'], 'source_repo': 'PiotrSty/impact-psnc-polish-ocr', 'source_revision': 'c7cb156fb95d2880699c33725bbaf1fbc1008fea', 'reference_status': 'automatic-count-matched-unreviewed'}, 'packages': ['transformers==4.57.6', 'tokenizers==0.22.2', 'huggingface_hub==0.36.2', 'pillow==11.3.0', 'bitsandbytes==0.48.1', 'accelerate==1.13.0', 'sentencepiece==0.2.1'], 'models': {'qwen3-vl-4b': {'repo': 'Qwen/Qwen3-VL-4B-Instruct', 'revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17', 'license': 'Apache-2.0', 'min_pixels': 262144, 'max_pixels': 1048576, 'max_new_tokens': 512, 'quantization': '4-bit NF4, double quantization, FP16 compute', 'attention': 'sdpa', 'prompt': 'Transcribe only the visible printed text in this historical Polish line crop. Preserve source spelling, accents, capitalization, punctuation and the historical characters long s (ſ), accented a (á) and e with stroke (ɇ) when visible. Do not modernize, translate, correct spelling, expand abbreviations or invent cut-off text. Return only the transcription as plain text, without commentary or Markdown.', 'packages': ['transformers==4.57.6', 'tokenizers==0.22.2', 'huggingface_hub==0.36.2', 'pillow==11.3.0', 'bitsandbytes==0.48.1', 'accelerate==1.13.0', 'sentencepiece==0.2.1']}, 'trocr-mixed-v3': {'repo': 'PiotrSty/trocr-pl-mixed-v3', 'revision': '85d0c91c26f8e088849096dded7c9ba10b4cd9c9', 'weights_sha256': '19437f5769b8df8be5af1fb0c5bbcc3acccae9d889f337f0c4b9e94fa80d8bb7', 'dtype': 'float32', 'max_new_tokens': 128, 'num_beams': 4, 'packages': ['transformers==4.57.6', 'tokenizers==0.22.2', 'huggingface_hub==0.36.2', 'pillow==11.3.0', 'bitsandbytes==0.48.1', 'accelerate==1.13.0', 'sentencepiece==0.2.1']}}, 'policy': {'reference_text_sent_to_models': False, 'automatic_training_promotion': False, 'human_crop_geometry_review_required': True, 'human_diplomatic_text_review_required': True, 'upstream_pretraining_overlap_audited': False, 'work_edition_near_duplicates_audited': False, 'sota_claim': False, 'new_line_profile_GPU_validated': False}, 'preparation_code_revision': '63a2a4d6682379f269d6d73940ae6f7986740679'}
CODE_REVISION = 'e24545b22b65d42937fd88714ff99c7b81e50f90'
WORK = Path('/content') / ('recognizer-data-v3-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
WORK.mkdir(parents=True)
repo = Path('/content/OCR_engine-recognizer-data-v3')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == CODE_REVISION
sys.path.insert(0, str(repo))
from training.recognizer_data_pilot import stage, combine, package, ENGINES
from training.full_page_pilot import digest, fetch, write_json, read_rows
write_json(WORK / 'config.json', CONFIG)
write_json(WORK / 'code-provenance.json', {'code_revision': CODE_REVISION,
    'runner_sha256': digest(repo / 'training/recognizer_data_pilot.py'),
    'qwen_runner_sha256': digest(repo / 'training/full_page_pilot.py'),
    'reference_text_sent_to_models': False, 'gpu_execution_validated_locally': False})
print('WORK:', WORK)


In [ ]:
archive = fetch(CONFIG, CONFIG['dataset']['path'], WORK / 'input.zip', CONFIG['dataset']['archive_sha256'])
dataset = WORK / 'dataset'
audit = stage(CONFIG, archive, dataset)
write_json(WORK / 'staging-report.json', audit)
print(json.dumps(audit, indent=2))


In [ ]:
import os
from pathlib import Path
import subprocess
import sys
import venv


def prepare_pilot_environment(environment):
    environment = Path(environment)
    # Colab can lack ensurepip; repair a partial venv without deleting its files.
    venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)
    python = environment / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    prefix = subprocess.check_output(
        [str(python), '-c', 'import sys; print(sys.prefix)'], text=True).strip()
    if Path(prefix).resolve() != environment.resolve():
        raise RuntimeError('Refusing installation outside the requested model environment.')
    return python


def pilot_pip_command(python, *arguments):
    # Host pip manages the target interpreter even when its venv has no local pip.
    return [sys.executable, '-m', 'pip', '--python', str(python), *arguments]

import traceback
python = None
bootstrap = {'status': 'error'}
try:
    import torch
    assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'
    candidate_python = prepare_pilot_environment('/content/recognizer-data-v3-env')
    command = pilot_pip_command(candidate_python, 'install', '--no-cache-dir', *CONFIG['packages'])
    installed = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    (WORK / 'installation.log').write_text(installed.stdout, encoding='utf-8')
    assert installed.returncode == 0, installed.stdout[-6000:]
    required = {item.split('==')[0]: item.split('==')[1] for item in CONFIG['packages']}
    preflight = ('import importlib.metadata,json,torch; required=' + repr(required) +
        '; actual={k:importlib.metadata.version(k) for k in required}; '
        'assert actual==required,(actual,required); '
        'from transformers import Qwen3VLForConditionalGeneration,TrOCRProcessor,VisionEncoderDecoderModel; '
        'from transformers.generation import GenerationMixin; '
        'assert torch.cuda.is_available(); '
        'print(json.dumps({"packages":actual,"torch":torch.__version__,"gpu":torch.cuda.get_device_name(0)}))')
    checked = subprocess.run([str(candidate_python), '-c', preflight], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
    (WORK / 'preflight.log').write_text(checked.stdout, encoding='utf-8')
    assert checked.returncode == 0, checked.stdout[-6000:]
    python = candidate_python
    bootstrap = {'status': 'ok', 'interpreter': str(python), 'gpu': torch.cuda.get_device_name(0)}
except Exception:
    bootstrap['error'] = traceback.format_exc()
    print(bootstrap['error'])
write_json(WORK / 'bootstrap.json', bootstrap)
print('BOOTSTRAP:', bootstrap['status'])


In [ ]:
runs = {}
for engine in ENGINES:
    if python is None:
        runs[engine] = {'status': 'not-started', 'reason': 'environment preflight failed'}
        continue
    command = [str(python), '-u', '-m', 'training.recognizer_data_pilot', '--config', str(WORK / 'config.json'),
               '--engine', engine, '--inputs', str(dataset / 'inference-inputs.jsonl'),
               '--output', str(WORK / 'predictions' / engine)]
    env = dict(os.environ, PYTHONPATH=str(repo), TOKENIZERS_PARALLELISM='false')
    with (WORK / (engine + '.log')).open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=repo, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='')
        code = process.wait()
    runs[engine] = {'exit_code': code, 'status': 'finished' if code == 0 else 'failed'}
    write_json(WORK / 'worker-exits.json', runs)
write_json(WORK / 'worker-exits.json', runs)


In [ ]:
try:
    report = combine(CONFIG, dataset, WORK / 'predictions', WORK / 'combined')
    print(json.dumps(report, ensure_ascii=False, indent=2))
except Exception:
    import traceback
    write_json(WORK / 'combine-error.json', {'error': traceback.format_exc()})
    print(traceback.format_exc())
result_archive = package(WORK)
print('EVIDENCE:', result_archive)


## Pobierz wynik

Przekaz plik `recognizer-data-v3-teacher-evidence.zip`. Nie zawiera wag ani skanow; obrazy mozna odtworzyc z przypietego pakietu HF. Bledy, brakujace odpowiedzi i limity pozostaja w raporcie.

In [ ]:
from google.colab import files
result_archive = package(WORK)
files.download(str(result_archive))
